# 🪞 entireHistoryOfYou — clone yourself on a free GPU

Fine-tune a small LLM (Qwen2.5-3B) on **your own chat history** so it texts like you — QLoRA, on a free Colab **T4**.

**Before anything:** `Runtime ▸ Change runtime type ▸ T4 GPU`.

> 🔒 Everything runs in *your* Colab session. Your chat data is uploaded only to this notebook's temporary VM and never leaves it.

## 0. Check you're on a GPU

In [ ]:
!nvidia-smi -L  # should print a Tesla T4 (or better)

## 1. Clone the project

Opening this notebook from GitHub loads **only the notebook** — not the repo's code. This cell clones the repo onto the runtime so `src/` and `configs/` are importable. (Re-runnable: it pulls the latest if already cloned.)

In [ ]:
import os
os.chdir("/content")
if not os.path.isdir("entireHistoryOfYou"):
    !git clone https://github.com/zeukid/entireHistoryOfYou.git
%cd /content/entireHistoryOfYou
!git pull -q
!ls

## 2. Install dependencies (pinned for T4)

In [ ]:
%pip install -q -r requirements.txt
print("\n✅ deps installed — if Colab asks to restart the session, do it, then re-run from cell 1.")

## 3. Upload your WhatsApp export

On your phone: open a chat ▸ ⋮ / contact name ▸ **Export chat** ▸ **Without media**. Upload the resulting `_chat.txt` (or `.txt`) here.

In [ ]:
import os
from google.colab import files

os.makedirs("exports", exist_ok=True)
up = files.upload()                      # choose your _chat.txt
EXPORT = os.path.join("exports", next(iter(up)))
os.replace(next(iter(up)), EXPORT)
print("Saved export to", EXPORT)

## 4. See who's in the chat

Note your exact name as it appears below.

In [ ]:
!python -m src.build_dataset --export "{EXPORT}" --list-senders

## 5. Build your training set

Set `ME` to your **exact** name from the list above, then run. Writes `data/train.jsonl` + `data/val.jsonl`.

In [ ]:
ME = "Your Name"   # <-- EDIT THIS
!python -m src.build_dataset --export "{EXPORT}" --me "{ME}" --out data/train.jsonl

## 6. Fine-tune (QLoRA)

Downloads Qwen2.5-3B (ungated — no token needed) on first run, then trains. A few thousand messages take ~15–40 min on a T4. Watch train + eval loss tick down.

In [ ]:
!python -m src.train --config configs/qwen3b_t4.yaml

## 7. Talk to your clone 🎉

Run, then call `reply("...")` with anything.

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import PeftModel

BASE, ADAPTER = "Qwen/Qwen2.5-3B-Instruct", "adapters/me"
bnb = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4",
                         bnb_4bit_compute_dtype=torch.float16, bnb_4bit_use_double_quant=True)
tok = AutoTokenizer.from_pretrained(ADAPTER)
model = AutoModelForCausalLM.from_pretrained(BASE, quantization_config=bnb, device_map="auto")
model = PeftModel.from_pretrained(model, ADAPTER)

def reply(msg, temperature=0.8):
    chat = [{"role": "user", "content": msg}]
    prompt = tok.apply_chat_template(chat, tokenize=False, add_generation_prompt=True)
    ins = tok(prompt, return_tensors="pt").to(model.device)
    out = model.generate(**ins, max_new_tokens=160, do_sample=True,
                         temperature=temperature, top_p=0.9)
    print(tok.decode(out[0][ins.input_ids.shape[1]:], skip_special_tokens=True))

reply("hey what are you up to this weekend?")